# 04 — Ablation, Sentetik Enjeksiyon, Kalibre Transfer ve USAD (Faz ML-3)

Beş soruya deneyle cevap verir:

1. **Ablation**: Model gerçekten *arızayı* mı öğreniyor, yoksa dataset artifact'ini mi?
   (ALFA airspeed±, UAV missingness±, residual-only vs diğerleri)
2. **Sentetik enjeksiyon**: Gerçek veride hiç olmayan arıza ailelerini (donma, bias, drift,
   gürültü, *stealthy* GPS spoofing, dropout) framework yakalıyor mu?
3. **H10 düzeltmesi**: Uçuş skoru `max(pencere)` yerine `eşik-üstü pencere oranı` olursa
   yanlış alarm düşer mi?
4. **Kalibre transfer**: SEAD'de eşikleri SEAD normalleriyle yeniden kalibre edince
   (ML-1'deki %100 yanlış alarmın karşı deneyi) transfer kurtulur mu?
5. **USAD**: Adversarial autoencoder LSTM-AE'yi geçer mi?

## Enjeksiyon matematiği (neden fiziksel olarak anlamlı?)

Her enjeksiyon, o arıza ailesinin literatürdeki imzasını üretir ($\sigma$ = sinyalin kendi std'si):

| Enjeksiyon | Model | Hedef imza |
|---|---|---|
| freeze | $x_t = x_{t_0}, \; t \ge t_0$ | rolling varyans → 0, `frozen_count` ↑ |
| bias | $x_t \mathrel{+}= 4\sigma$ | CUSUM adım tespiti, residual kayması |
| drift | $x_t \mathrel{+}= 3\sigma \cdot \frac{t-t_0}{60}$ | CUSUM'un var oluş nedeni: yavaş birikim |
| noise | $x_t \mathrel{+}= \mathcal{N}(0, (3\sigma)^2)$ | rolling std/RMS, spektral enerji ↑ |
| gps_ramp | $lat_t \mathrel{+}= \frac{2 \cdot (t-t_0)}{111320}$ (2 m/s kuzey) | **stealthy spoofing**: sıçrama yok; `gps_speed_residual` (receiver hızı sabit!) |
| dropout | attitude bloğu → NaN | missingness/stale imzası, H3'ün kontrollü versiyonu |

**Kural**: enjekte uçuşlar yalnızca **test**te; eğitim ve eşikler tamamen temiz normal veriden.

## USAD matematiği

Tek AE'nin zaafı: anomaliye de "yakın bir normal" üretip düşük hata verebilir. USAD, encoder $E$ +
iki decoder ($D_1, D_2$) ile iki fazlı eğitim yapar; $AE_1 = D_1 \circ E$, $AE_2 = D_2 \circ E$:

$$\mathcal{L}_{AE_1} = \tfrac{1}{n}\|W - AE_1(W)\|^2 + (1-\tfrac{1}{n})\|W - AE_2(AE_1(W))\|^2$$
$$\mathcal{L}_{AE_2} = \tfrac{1}{n}\|W - AE_2(W)\|^2 - (1-\tfrac{1}{n})\|W - AE_2(AE_1(W))\|^2$$

($n$ = epoch numarası.) $AE_2$ zamanla $AE_1$'in *sahte* reconstruction'larını gerçekten ayırt etmeyi
öğrenir (adversarial); skor iki hatanın karışımı:

$$s(W) = \alpha\|W - AE_1(W)\|^2 + \beta\|W - AE_2(AE_1(W))\|^2, \quad \alpha{=}\beta{=}0.5$$

Anomalilerde $AE_1$ kötü üretir → $AE_2$ zinciri hatayı **büyütür** (tek AE'de olmayan kaldıraç).


In [1]:
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

FEAT = ROOT / "data/gold/ml_features"
manifest = json.loads((FEAT / "split_manifest.json").read_text(encoding="utf-8"))

from src.ml.data.scaling import apply_scaler_params
from src.ml.data.windowing import build_windows
from src.ml.features.alfa_features import build_alfa_features
from src.ml.features.uav_attack_features import MISSINGNESS_COLUMNS, build_px4_features, build_uav_attack_features
from src.ml import injection as inj

torch.set_num_threads(4)
NORMALS = {"normal", "benign"}

MODULES = {  # notebook 02 ile ayni
    "alfa": {
        "kontrol_tepki": ["abs_roll_error", "abs_pitch_error", "abs_yaw_error", "roll_error_rate",
                           "pitch_error_rate", "yaw_error_rate", "roll_error_2s_rms", "roll_error_5s_rms",
                           "pitch_error_2s_rms", "pitch_error_5s_rms", "yaw_error_5s_rms",
                           "roll_spec_energy_5s", "pitch_spec_energy_5s", "roll_error_cusum_pos",
                           "pitch_error_cusum_pos", "turn_residual", "turn_residual_5s_rms"],
        "rehberlik": ["alt_error", "aspd_error", "xtrack_error", "path_dev_mag", "wp_dist",
                       "alt_error_cusum_pos", "alt_error_cusum_neg", "xtrack_error_cusum_pos",
                       "climb_residual", "energy_rate", "altitude_rate", "airspeed_error",
                       "abs_airspeed_error", "xtrack_error_5s_rms"],
    },
    "uav_attack": {
        "nav_butunlugu": ["gps_step_m", "log_gps_speed", "gps_accel_mps2", "vertical_rate_calc",
                           "gps_speed_residual", "vertical_rate_residual", "course_change_deg",
                           "gps_step_5s_max", "gps_step_5s_rms", "gps_speed_residual_cusum_pos",
                           "gps_frozen_count"],
        "sinyal_kalitesi": ["jamming_indicator", "noise_per_ms", "hdop", "vdop", "satellites_used",
                             "s_variance_m_s", "eph", "epv", "jamming_1s_mean", "jamming_5s_max",
                             "noise_5s_std", "noise_5s_mean", "hdop_5s_max", "sats_5s_min",
                             "hdop_cusum_pos", "noise_per_ms_cusum_pos"],
        "veri_kalitesi": ["attitude_missing", "battery_missing", "gps_health_missing",
                           "num_missing_groups", "attitude_stale_count", "attitude_stale_s"],
    },
}

def load_feats(name):
    df = pd.read_parquet(FEAT / name / f"{name}_ml_features.parquet")
    if name == "alfa":
        df = df[df["label"] != "unknown"]
    scaler = json.loads((ROOT / "artifacts/scalers" / f"{name}_robust_scaler.json").read_text())
    return df, scaler

def anomaly_scores(model, X):
    return -model.score_samples(X)

def fit_modules(scaled, split, modules, seed=0):
    """Temiz train ucuslarinda modul IF'lerini egitir; val'den ucus/satir esikleri."""
    tr = scaled[scaled["source_id"].isin(split["train"])]
    va = scaled[scaled["source_id"].isin(split["val"])]
    fitted = {}
    for mod, cols in modules.items():
        cols = [c for c in cols if c in scaled.columns]
        m = IsolationForest(n_estimators=300, max_samples=256, random_state=seed, n_jobs=-1).fit(tr[cols])
        va_s = anomaly_scores(m, va[cols])
        tau_flight = va.assign(s=va_s).groupby("source_id")["s"].max().max()
        tau_row = float(np.quantile(va_s, 0.99))
        fitted[mod] = (m, cols, tau_flight, tau_row)
    return fitted

def fused_flight_ratios(fitted, df_scaled):
    ratios = {}
    for mod, (m, cols, tau_f, _) in fitted.items():
        s = anomaly_scores(m, df_scaled[cols])
        ratios[mod] = df_scaled.assign(s=s).groupby("source_id")["s"].max() / tau_f
    R = pd.DataFrame(ratios)
    R["fusion"] = R.max(axis=1)
    return R

print("hazir")

hazir


## 1. Ablation matrisi — model neyi öğreniyor?

IF-füzyon (hızlı olduğu için ablation aracı), 5 seed ortalaması, uçuş-ROC + tip tespiti.

In [2]:
def run_fusion_variant(name, modules, drop_cols=frozenset()):
    df, scaler = load_feats(name)
    labels = manifest["sources"][name]["flight_labels"]
    scaled = apply_scaler_params(df, scaler)
    mods = {mod: [c for c in cols if c not in drop_cols] for mod, cols in modules.items()}
    mods = {mod: cols for mod, cols in mods.items() if cols}
    rocs, dets = [], []
    for split_name, split in manifest["sources"][name]["splits"].items():
        fitted = fit_modules(scaled, split, mods, seed=split["seed"])
        te = scaled[scaled["source_id"].isin(split["test"])]
        R = fused_flight_ratios(fitted, te)
        y = np.array([0 if labels[f] == "normal" else 1 for f in R.index])
        rocs.append(roc_auc_score(y, R["fusion"]))
        dets.append(float((R["fusion"][y == 1] > 1).mean()))
    return np.mean(rocs), np.std(rocs), np.mean(dets)

AIRSPEED_COLS = frozenset(["airspeed_error", "abs_airspeed_error", "aspd_error", "airspeed_rate",
                            "airspeed_error_5s_mean", "airspeed_error_5s_std", "airspeed_error_5s_max",
                            "airspeed_error_5s_rms", "airspeed_available"])
RESIDUAL_ONLY_DROP = frozenset(  # residual OLMAYANLARI dusur: rate/spec/rolling turevleri kalir mi? tersini yap:
    [])  # asagida ayri ele aliniyor

rows = []
for variant, (name, mods, drop) in {
    "ALFA / tam": ("alfa", MODULES["alfa"], frozenset()),
    "ALFA / airspeed'siz": ("alfa", MODULES["alfa"], AIRSPEED_COLS),
    "ALFA / sadece-rehberlik": ("alfa", {"rehberlik": MODULES["alfa"]["rehberlik"]}, frozenset()),
    "ALFA / sadece-kontrol": ("alfa", {"kontrol_tepki": MODULES["alfa"]["kontrol_tepki"]}, frozenset()),
    "UAV / tam": ("uav_attack", MODULES["uav_attack"], frozenset()),
    "UAV / missingness'siz": ("uav_attack",
        {m: c for m, c in MODULES["uav_attack"].items() if m != "veri_kalitesi"}, frozenset()),
    "UAV / sadece-nav": ("uav_attack", {"nav_butunlugu": MODULES["uav_attack"]["nav_butunlugu"]}, frozenset()),
}.items():
    roc_m, roc_s, det = run_fusion_variant(name, mods, drop)
    rows.append({"varyant": variant, "ucus_roc": round(roc_m, 3), "roc_std": round(roc_s, 3),
                 "tespit@1": round(det, 3)})
abl = pd.DataFrame(rows).set_index("varyant")
print(abl.to_string())

                         ucus_roc  roc_std  tespit@1
varyant                                             
ALFA / tam                  0.833    0.153     0.778
ALFA / airspeed'siz         0.808    0.121     0.744
ALFA / sadece-rehberlik     0.864    0.072     0.744
ALFA / sadece-kontrol       0.511    0.256     0.206
UAV / tam                   0.600    0.189     0.569
UAV / missingness'siz       0.423    0.129     0.508
UAV / sadece-nav            0.554    0.075     0.415


In [3]:
# missingness ablation'inin ping_dos'a etkisi (kritik kisayol sorusu)
def dos_detection(mods_subset):
    df, scaler = load_feats("uav_attack")
    labels = manifest["sources"]["uav_attack"]["flight_labels"]
    scaled = apply_scaler_params(df, scaler)
    dets = []
    for split_name, split in manifest["sources"]["uav_attack"]["splits"].items():
        fitted = fit_modules(scaled, split, mods_subset, seed=split["seed"])
        te = scaled[scaled["source_id"].isin(split["test"])]
        R = fused_flight_ratios(fitted, te)
        dos = [f for f in R.index if labels[f] == "ping_dos"]
        dets.append(float((R.loc[dos, "fusion"] > 1).mean()))
    return np.mean(dets)

full = dos_detection(MODULES["uav_attack"])
nomiss = dos_detection({m: c for m, c in MODULES["uav_attack"].items() if m != "veri_kalitesi"})
print(f"ping_dos tespiti: missingness dahil={full:.2f}, haric={nomiss:.2f}")
print("Fark buyukse: DoS 'tespitimiz' buyuk olcude telemetry-availability imzasi demektir --")
print("rapor dili 'cyberattack dynamics' degil 'telemetry availability signature' olmali (FableChat uyarisi).")

ping_dos tespiti: missingness dahil=0.37, haric=0.37
Fark buyukse: DoS 'tespitimiz' buyuk olcude telemetry-availability imzasi demektir --
rapor dili 'cyberattack dynamics' degil 'telemetry availability signature' olmali (FableChat uyarisi).


## 2. Sentetik enjeksiyon — temiz modelle boz-ve-ölç

Split_00'ın **temiz** train'iyle eğitilen modüller, normal uçuşların bozulmuş kopyalarını puanlıyor.
Enjeksiyon Silver düzeyinde yapılır, feature'lar **yeniden üretilir** (rolling/CUSUM arızayı doğal akışta görür).

In [4]:
ALFA_SILVER = pd.read_parquet(ROOT / "data/silver/alfa_silver.parquet")
UAV_SILVER = pd.read_parquet(ROOT / "data/silver/uav_attack_silver.parquet")

INJECTIONS = {
    "alfa": {
        "freeze(roll)": lambda g, r: inj.inject_freeze(g, "roll_measured", rng=r),
        "bias(pitch)": lambda g, r: inj.inject_bias(g, "pitch_measured", rng=r),
        "drift(alt)": lambda g, r: inj.inject_drift(g, "alt", "ts_ns", rng=r),
        "noise(roll)": lambda g, r: inj.inject_noise(g, "roll_measured", rng=r),
    },
    "uav_attack": {
        "gps_ramp(2m/s)": lambda g, r: inj.inject_gps_ramp(g, meters_per_s=2.0, rng=r),
        "dropout(attitude)": lambda g, r: inj.inject_dropout(g, ["roll_deg", "pitch_deg", "yaw_deg"], rng=r),
        "noise(noise_per_ms)": lambda g, r: inj.inject_noise(g, "noise_per_ms", rng=r),
        "freeze(lat/lon)": lambda g, r: inj.inject_freeze(inj.inject_freeze(g, "lat", rng=r), "lon", rng=r),
    },
}
BUILDERS = {"alfa": build_alfa_features, "uav_attack": build_uav_attack_features}

def injection_experiment(name, silver):
    df, scaler = load_feats(name)
    split = manifest["sources"][name]["splits"]["split_00"]
    labels = manifest["sources"][name]["flight_labels"]
    scaled = apply_scaler_params(df, scaler)
    fitted = fit_modules(scaled, split, MODULES[name], seed=0)

    # hedef: train'e girmemis normal ucuslar (val + test-normal)
    targets = split["val"] + split["test_normal"]
    rng = np.random.default_rng(42)
    rows = []
    for inj_name, fn in INJECTIONS[name].items():
        for sid in targets:
            g = silver[silver["source_id"] == sid].sort_values(
                "ts_ns" if name == "alfa" else "timestamp").reset_index(drop=True)
            if g.empty:
                continue
            feats = BUILDERS[name](fn(g, rng))
            fscaled = apply_scaler_params(feats, scaler)
            # ucus-duzeyi fuzyon orani
            ratios = {}
            row_alarm_t = np.inf
            onset_t = feats[feats["label"].str.startswith("inj_")]["t_rel_s"].min()
            for mod, (m, cols, tau_f, tau_row) in fitted.items():
                cols_p = [c for c in cols if c in fscaled.columns]
                s = anomaly_scores(m, fscaled[cols_p])
                ratios[mod] = float(np.max(s)) / tau_f
                alarms = fscaled.assign(s=s)
                hit = alarms[(alarms["t_rel_s"] >= onset_t) & (alarms["s"] > tau_row)]
                if len(hit):
                    row_alarm_t = min(row_alarm_t, hit["t_rel_s"].iloc[0])
            fusion = max(ratios.values())
            rows.append({"enjeksiyon": inj_name, "ucus": sid, "fusion": round(fusion, 2),
                         "tespit": fusion > 1,
                         "gecikme_s": round(row_alarm_t - onset_t, 1) if np.isfinite(row_alarm_t) else np.nan,
                         "baskin_modul": max(ratios, key=ratios.get)})
    return pd.DataFrame(rows)

inj_a = injection_experiment("alfa", ALFA_SILVER)
inj_u = injection_experiment("uav_attack", UAV_SILVER)
for name, t in [("ALFA", inj_a), ("UAV Attack", inj_u)]:
    print(f"=== {name} enjeksiyon sonuclari ===")
    print(t.groupby("enjeksiyon").agg(tespit_orani=("tespit", "mean"),
                                       ort_gecikme_s=("gecikme_s", "mean"),
                                       baskin_modul=("baskin_modul", lambda x: x.mode().iloc[0])).round(2).to_string())
    print()

=== ALFA enjeksiyon sonuclari ===
              tespit_orani  ort_gecikme_s   baskin_modul
enjeksiyon                                              
bias(pitch)           0.25          13.40  kontrol_tepki
drift(alt)            0.75           0.28      rehberlik
freeze(roll)          0.25          29.95      rehberlik
noise(roll)           0.75           7.17  kontrol_tepki

=== UAV Attack enjeksiyon sonuclari ===
                     tespit_orani  ort_gecikme_s     baskin_modul
enjeksiyon                                                       
dropout(attitude)             0.0         447.20    nav_butunlugu
freeze(lat/lon)               0.0           0.00  sinyal_kalitesi
gps_ramp(2m/s)                0.0          43.15  sinyal_kalitesi
noise(noise_per_ms)           0.0         447.20    nav_butunlugu



### Enjeksiyonun kritik sorusu — stealthy `gps_ramp`

ML-1'de kaba SITL spoofing (78 km sıçrama) trivial'di, live/hackrf spoofing'i `gps_speed_residual` yakaladı.
`gps_ramp(2 m/s)` bunun kontrollü versiyonu: konum yavaşça kayar, receiver hızı **değişmez**.
Tespit + baskın modül `nav_butunlugu` ise analytical-redundancy tezi enjeksiyonla da doğrulanmış olur.

## 3. H10 düzeltmesi — uçuş skoru: max vs eşik-üstü oran

`max` tek uç pencereye teslim olur: N pencere için $P(\max > Q_{99}) \approx 1-(0.99)^N \to 1$.
**Eşik-üstü oran** $\rho(u) = \frac{1}{N}\sum \mathbb{1}[s_i > \tau_{row}]$ ise normalde beklenen değeri
~0.01'e çakılıdır ve N ile şişmez — anomalili uçuşta imza pencereleri oranı yükseltir.

In [5]:
from src.ml.data.windowing import build_windows as _bw

class LSTMAE(nn.Module):
    def __init__(self, f, hidden=32, latent=16):
        super().__init__()
        self.encoder = nn.LSTM(f, hidden, batch_first=True)
        self.to_latent = nn.Linear(hidden, latent)
        self.from_latent = nn.Linear(latent, hidden)
        self.decoder = nn.LSTM(hidden, hidden, batch_first=True)
        self.head = nn.Linear(hidden, f)
    def forward(self, x):
        _, (h, _) = self.encoder(x)
        rep = self.from_latent(self.to_latent(h[-1])).unsqueeze(1).repeat(1, x.shape[1], 1)
        dec, _ = self.decoder(rep)
        return self.head(dec)

def masked_mse(x, xhat, m, per_sample=False):
    se = ((x - xhat) ** 2) * m
    if per_sample:
        return se.sum(dim=(1, 2)) / m.sum(dim=(1, 2)).clamp(min=1.0)
    return se.sum() / m.sum().clamp(min=1.0)

def train_ae(model, Xtr, Mtr, Xva, Mva, *, seed, epochs=40, batch=64, lr=1e-3, patience=5):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xtr_t, Mtr_t = torch.tensor(Xtr), torch.tensor(Mtr)
    Xva_t, Mva_t = torch.tensor(Xva), torch.tensor(Mva)
    best, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), batch):
            idx = perm[i:i+batch]
            loss = masked_mse(Xtr_t[idx], model(Xtr_t[idx]), Mtr_t[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = float(masked_mse(Xva_t, model(Xva_t), Mva_t))
        if vl < best - 1e-5:
            best, best_state, bad = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model

def window_scores(model, X, M, batch=512):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            out.append(masked_mse(torch.tensor(X[i:i+batch]), model(torch.tensor(X[i:i+batch])),
                                  torch.tensor(M[i:i+batch]), per_sample=True).numpy())
    return np.concatenate(out) if out else np.array([])

AE_FEATURES = {  # notebook 03 ile ayni
    "alfa": ["abs_roll_error", "abs_pitch_error", "abs_yaw_error", "roll_error_rate",
              "pitch_error_rate", "yaw_error_rate", "roll_error_2s_rms", "roll_error_5s_rms",
              "pitch_error_2s_rms", "pitch_error_5s_rms", "yaw_error_5s_rms",
              "roll_spec_energy_5s", "pitch_spec_energy_5s", "turn_residual",
              "alt_error", "aspd_error", "xtrack_error", "path_dev_mag", "climb_residual",
              "energy_rate", "altitude_rate", "abs_airspeed_error", "gps_speed_calc_mps"],
    "uav_attack": ["gps_step_m", "log_gps_speed", "gps_accel_mps2", "vertical_rate_calc",
                    "gps_speed_residual", "vertical_rate_residual", "course_change_deg",
                    "gps_frozen_count", "jamming_indicator", "noise_per_ms", "hdop", "vdop",
                    "satellites_used", "s_variance_m_s", "eph", "epv",
                    "roll_rate", "pitch_rate", "yaw_rate",
                    "attitude_missing", "num_missing_groups", "attitude_stale_s"],
}
WPAR = {"alfa": (40, 4), "uav_attack": (50, 5)}

def load_win(name):
    df, scaler = load_feats(name)
    cols = [c for c in AE_FEATURES[name] if c in df.columns]
    scaled = apply_scaler_params(df, scaler)
    mask_df = df[cols].notna()
    sv = scaled[["source_id", "t_rel_s", "label"]].copy()
    for c in cols:
        sv[c] = scaled[c].where(mask_df[c])
    w, s = WPAR[name]
    return _bw(sv, cols, window=w, stride=s, max_gap_s=2.0)

def h10_compare(name):
    X, M, meta = load_win(name)
    labels = manifest["sources"][name]["flight_labels"]
    rows = []
    for split_name, split in manifest["sources"][name]["splits"].items():
        tr_i = meta["source_id"].isin(split["train"]).to_numpy()
        va_i = meta["source_id"].isin(split["val"]).to_numpy()
        te_i = meta["source_id"].isin(split["test"]).to_numpy()
        model = train_ae(LSTMAE(X.shape[2]), X[tr_i], M[tr_i], X[va_i], M[va_i], seed=split["seed"])
        s_va, s_te = window_scores(model, X[va_i], M[va_i]), window_scores(model, X[te_i], M[te_i])
        tau_row = float(np.quantile(s_va, 0.99))
        te_meta = meta[te_i].assign(s=s_te)
        g = te_meta.groupby("source_id")["s"]
        for agg_name, fs in [("max", g.max()), ("oran>tau", g.apply(lambda x: (x > tau_row).mean()))]:
            yf = np.array([0 if labels[f] == "normal" else 1 for f in fs.index])
            # esik: val ucuslarindaki ayni aggregasyonun maksimumu
            va_meta = meta[va_i].assign(s=s_va)
            gv = va_meta.groupby("source_id")["s"]
            tau_f = (gv.max() if agg_name == "max" else gv.apply(lambda x: (x > tau_row).mean())).max()
            tau_f = max(tau_f, 1e-6)
            rows.append({"split": split_name, "skor": agg_name,
                         "ucus_roc": roc_auc_score(yf, fs.values),
                         "tespit": float((fs[yf == 1] > tau_f).mean()),
                         "yanlis_alarm": float((fs[yf == 0] > tau_f).mean()) if (yf == 0).any() else np.nan})
    r = pd.DataFrame(rows)
    return r.groupby("skor")[["ucus_roc", "tespit", "yanlis_alarm"]].agg(["mean", "std"]).round(3)

for name in ["alfa", "uav_attack"]:
    print(f"=== {name} — LSTM-AE ucus skoru: max vs esik-ustu-oran (5 seed) ===")
    print(h10_compare(name).to_string(), "\n")

=== alfa — LSTM-AE ucus skoru: max vs esik-ustu-oran (5 seed) ===


         ucus_roc        tespit        yanlis_alarm       
             mean    std   mean    std         mean    std
skor                                                      
max         0.742  0.146  0.656  0.265          0.5  0.354
oran>tau    0.417  0.212  0.594  0.320          0.5  0.354 

=== uav_attack — LSTM-AE ucus skoru: max vs esik-ustu-oran (5 seed) ===


         ucus_roc        tespit        yanlis_alarm       
             mean    std   mean    std         mean    std
skor                                                      
max         0.677  0.126  0.785  0.034          0.6  0.548
oran>tau    0.515  0.321  0.785  0.034          0.8  0.447 



## 4. SEAD kalibre transfer — ML-1'in karşı deneyi

ML-1'de UAV eşikleriyle SEAD: normal yanlış alarm **1.0** (tam çöküş). Şimdi aynı modeller,
ama $\tau_m$'ler **SEAD'in kendi train-normal uçuşlarından** (split_00) yeniden kalibre ediliyor —
FableChat'in "yeni platformda 10-30 dk normal uçuşla kalibrasyon" tezinin testi.
Ayrım: **feature semantiği + model transfer edilir, eşik platforma aittir.**

In [6]:
df_u, scaler_u = load_feats("uav_attack")
df_s, _ = load_feats("uav_sead")
labels_s = manifest["sources"]["uav_sead"]["flight_labels"]
split_u = manifest["sources"]["uav_attack"]["splits"]["split_00"]
split_s = manifest["sources"]["uav_sead"]["splits"]["split_00"]

scaled_u = apply_scaler_params(df_u, scaler_u)
scaled_s = apply_scaler_params(df_s, scaler_u)  # scaler hala UAV'nin (model onunla egitildi)

tr_u = scaled_u[scaled_u["source_id"].isin(split_u["train"])]
cal = scaled_s[scaled_s["source_id"].isin(split_s["train"])]   # SEAD kalibrasyon normalleri
te_s = scaled_s[scaled_s["source_id"].isin(split_s["test"])]

ratios = {}
for mod, cols in MODULES["uav_attack"].items():
    cols = [c for c in cols if c in scaled_s.columns]
    m = IsolationForest(n_estimators=300, max_samples=256, random_state=0, n_jobs=-1).fit(tr_u[cols])
    tau_cal = cal.assign(s=anomaly_scores(m, cal[cols])).groupby("source_id")["s"].max().max()
    ratios[mod] = te_s.assign(s=anomaly_scores(m, te_s[cols])).groupby("source_id")["s"].max() / tau_cal
R = pd.DataFrame(ratios); R["fusion"] = R.max(axis=1)
R["label"] = [labels_s[f] for f in R.index]; R["y"] = (R["label"] != "normal").astype(int)
print("SEAD KALIBRE transfer (UAV modeli + SEAD esigi):")
print("  ucus ROC:", round(roc_auc_score(R["y"], R["fusion"]), 3), "(ML-1 kalibrasyonsuz: 0.375)")
print("  normal yanlis alarm@1:", round(float((R[R['y']==0]['fusion'] > 1).mean()), 2), "(ML-1: 1.00)")
print("  tip bazinda tespit@1:")
print(R[R["y"] == 1].groupby("label")["fusion"].apply(lambda x: (x > 1).mean()).round(2).to_string())

SEAD KALIBRE transfer (UAV modeli + SEAD esigi):
  ucus ROC: 0.783 (ML-1 kalibrasyonsuz: 0.375)
  normal yanlis alarm@1: 0.33 (ML-1: 1.00)
  tip bazinda tespit@1:
label
altitude_anomaly             0.4
external_position_anomaly    0.7
global_position_anomaly      1.0
mechanical_fault             0.6


## 5. USAD — adversarial autoencoder

In [7]:
class USAD(nn.Module):
    def __init__(self, w, f, latent=16):
        super().__init__()
        d = w * f
        self.E = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, latent), nn.ReLU())
        self.D1 = nn.Sequential(nn.Linear(latent, 64), nn.ReLU(), nn.Linear(64, d))
        self.D2 = nn.Sequential(nn.Linear(latent, 64), nn.ReLU(), nn.Linear(64, d))
        self.w, self.f = w, f
    def ae1(self, x): return self.D1(self.E(x.flatten(1))).view(-1, self.w, self.f)
    def ae2(self, x): return self.D2(self.E(x.flatten(1))).view(-1, self.w, self.f)

def train_usad(X, M, Xva, Mva, *, seed, epochs=40, batch=64, lr=1e-3):
    torch.manual_seed(seed)
    model = USAD(X.shape[1], X.shape[2])
    opt1 = torch.optim.Adam(list(model.E.parameters()) + list(model.D1.parameters()), lr=lr)
    opt2 = torch.optim.Adam(list(model.E.parameters()) + list(model.D2.parameters()), lr=lr)
    Xt, Mt = torch.tensor(X), torch.tensor(M)
    for ep in range(1, epochs + 1):
        w1, w2 = 1.0 / ep, 1.0 - 1.0 / ep
        perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), batch):
            idx = perm[i:i+batch]
            x, m = Xt[idx], Mt[idx]
            # faz 1: AE1
            x1 = model.ae1(x); x2_adv = model.ae2(x1)
            l1 = w1 * masked_mse(x, x1, m) + w2 * masked_mse(x, x2_adv, m)
            opt1.zero_grad(); l1.backward(); opt1.step()
            # faz 2: AE2 (adversarial: zincir hatasini BUYUTMEYI ogrenir)
            x1 = model.ae1(x).detach(); x2 = model.ae2(x); x2_adv = model.ae2(x1)
            l2 = w1 * masked_mse(x, x2, m) - w2 * masked_mse(x, x2_adv, m)
            opt2.zero_grad(); l2.backward(); opt2.step()
    return model

def usad_scores(model, X, M, alpha=0.5, beta=0.5, batch=512):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            x, m = torch.tensor(X[i:i+batch]), torch.tensor(M[i:i+batch])
            x1 = model.ae1(x); x2 = model.ae2(x1)
            s = alpha * masked_mse(x, x1, m, per_sample=True) + beta * masked_mse(x, x2, m, per_sample=True)
            out.append(s.numpy())
    return np.concatenate(out)

def run_usad(name):
    X, M, meta = load_win(name)
    labels = manifest["sources"][name]["flight_labels"]
    rows = []
    for split_name, split in manifest["sources"][name]["splits"].items():
        tr_i = meta["source_id"].isin(split["train"]).to_numpy()
        va_i = meta["source_id"].isin(split["val"]).to_numpy()
        te_i = meta["source_id"].isin(split["test"]).to_numpy()
        model = train_usad(X[tr_i], M[tr_i], X[va_i], M[va_i], seed=split["seed"])
        s_va, s_te = usad_scores(model, X[va_i], M[va_i]), usad_scores(model, X[te_i], M[te_i])
        tau_row = float(np.quantile(s_va, 0.99))
        te_meta = meta[te_i].assign(s=s_te)
        fs = te_meta.groupby("source_id")["s"].apply(lambda x: (x > tau_row).mean())  # H10 skoru
        yf = np.array([0 if labels[f] == "normal" else 1 for f in fs.index])
        rows.append({"split": split_name, "ucus_roc": roc_auc_score(yf, fs.values)})
    return pd.DataFrame(rows)["ucus_roc"]

for name in ["alfa", "uav_attack"]:
    r = run_usad(name)
    print(f"{name}: USAD ucus-ROC (oran skoru, 5 seed) = {r.mean():.3f} +- {r.std():.3f}")

alfa: USAD ucus-ROC (oran skoru, 5 seed) = 0.450 +- 0.145


uav_attack: USAD ucus-ROC (oran skoru, 5 seed) = 0.531 +- 0.296


## 6. Sonuç — ML-3 kapanışı

Bu notebook'un çıktıları `docs/ML1_BULGULAR_VE_HATALAR.md`'ye işlenecek. Karar tablosu:

- **Ablation** → hangi feature grubunun performansı taşıdığı ve missingness kısayolunun payı netleşti.
- **Enjeksiyon** → gerçek veride olmayan aileler (freeze/bias/drift/noise/stealthy-ramp/dropout) için
  tespit oranı + gecikme raporlandı; kapsam tablosunun "enjeksiyonla kapatıldı" sütunu doldu.
- **H10** → uçuş skoru `eşik-üstü oran` yanlış alarmı düşürüyorsa varsayılan skor bu olur.
- **Kalibre transfer** → framework'ün taşınabilirlik iddiası artık ölçülü: model+feature taşınır, eşik platforma aittir.
- **USAD** → LSTM-AE'ye karşı konumu netleşti; kazanan ML-4 (gerçek-zamanlı entegrasyon) modeli olur.
